# MCP Connectivity Tests

Validate connectivity and tool discovery for the remote MCP servers using the official MCP Python SDK.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project Root:", PROJECT_ROOT)

Project Root: c:\Users\praya\Desktop\Tarvel_Agent\trip_planner


In [2]:
from config.settings import settings

SERVERS = [
    {"name": "Kiwi MCP", "url": settings.kiwi_mcp_server_url},
    {"name": "Agentorist MCP", "url": settings.agentorist_mcp_server_url},
]

for server in SERVERS:
    print(f"{server['name']}: {server['url']}")

Kiwi MCP: https://mcp.kiwi.com
Agentorist MCP: https://mcp.agentorist.com/mcp


In [3]:
import asyncio

from mcp import ClientSession
from mcp.client.sse import sse_client
from mcp.client.streamable_http import streamable_http_client


async def list_tools_for_url(url: str) -> list[str]:
    normalized = url.rstrip("/")
    if normalized.endswith("/sse"):
        async with sse_client(url) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [tool.name for tool in tools.tools]

    async with streamable_http_client(url) as (read_stream, write_stream, _):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tools = await session.list_tools()
            return [tool.name for tool in tools.tools]


async def test_server(server: dict) -> dict:
    result = {
        "server": server["name"],
        "url": server["url"],
        "status": "unknown",
        "tools": [],
    }

    try:
        tools = await list_tools_for_url(server["url"])
        result["status"] = "connected"
        result["tools"] = tools
    except Exception as exc:
        result["status"] = f"failed: {exc}"

    return result


async def run_tests() -> list[dict]:
    results = []
    for server in SERVERS:
        results.append(await test_server(server))
    return results

In [4]:
results = await run_tests()

for result in results:
    print("=" * 60)
    print("Server:", result["server"])
    print("URL:", result["url"])
    print("Connection:", result["status"])
    print("Available tools:", result["tools"])

Server: Kiwi MCP
URL: https://mcp.kiwi.com
Connection: connected
Available tools: ['search-flight', 'feedback-to-devs']
Server: Agentorist MCP
URL: https://mcp.agentorist.com/mcp
Connection: connected
Available tools: ['list_verticals', 'search', 'search_all', 'find_options', 'book', 'list_venues', 'request_unsupported_booking']


In [19]:
import asyncio
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

async def test_kiwi():
    async with streamable_http_client("https://mcp.kiwi.com") as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()

            result = await session.call_tool(
                "search-flight",
                {
                    "flyFrom": "DEL",
                    "flyTo": "BOM",
                    "departureDate": "15/08/2026",
                    "curr": "INR",
                    "sortBy": "price"
                }
            )

            print(result)

await test_kiwi()

meta=None content=[TextContent(type='text', text='[\n  {\n    "flyFrom": "DEL",\n    "flyTo": "BOM",\n    "cityFrom": "New Delhi",\n    "cityTo": "Mumbai",\n    "departure": {\n      "utc": "2026-08-14T22:30:00.000Z",\n      "local": "2026-08-15T04:00:00.000"\n    },\n    "arrival": {\n      "utc": "2026-08-15T00:45:00.000Z",\n      "local": "2026-08-15T06:15:00.000"\n    },\n    "totalDurationInSeconds": 8100,\n    "durationInSeconds": 8100,\n    "price": 7085,\n    "deepLink": "https://on.kiwi.com/0nSVe2",\n    "currency": "INR"\n  },\n  {\n    "flyFrom": "DEL",\n    "flyTo": "BOM",\n    "cityFrom": "New Delhi",\n    "cityTo": "Mumbai",\n    "departure": {\n      "utc": "2026-08-15T03:00:00.000Z",\n      "local": "2026-08-15T08:30:00.000"\n    },\n    "arrival": {\n      "utc": "2026-08-15T05:15:00.000Z",\n      "local": "2026-08-15T10:45:00.000"\n    },\n    "totalDurationInSeconds": 8100,\n    "durationInSeconds": 8100,\n    "price": 7704,\n    "deepLink": "https://on.kiwi.com/GURo

In [22]:
from pprint import pprint
from tools.hotel_tools import search_local_places

result = search_local_places("Mumbai", venue="DY Patil Stadium")

pprint(result)

Agentorist MCP payload for search: {'vertical': 'hotels', 'query': 'best hotels', 'location': 'Mumbai', 'agent_client': 'TripPlanner'}
{'available_tools': ['list_verticals',
                     'search',
                     'search_all',
                     'find_options',
                     'book',
                     'list_venues',
                     'request_unsupported_booking'],
 'data': {'content': [{'text': '{"vertical":"","results":[],"result_count":0,"bookable_count":0,"error":"internal_error","error_message":"An '
                               'internal error occurred. Please retry."}',
                       'type': 'text'}],
          'structured': {'bookable_count': 0,
                         'error': 'internal_error',
                         'error_message': 'An internal error occurred. Please '
                                          'retry.',
                         'result_count': 0,
                         'results': [],
                         'vertic

In [32]:
import asyncio

from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

async def test_local_search():
    async with streamable_http_client("https://mcp.agentorist.com/mcp") as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()

            result = await session.call_tool(
                "search",
                {
                    "vertical": "local",
                    "query": "restaurants",
                    "location": "Miami",
                    "agent_client": "TripPlanner"
                }
            )

            print(result)

await test_local_search()

meta=None content=[TextContent(type='text', text='{"vertical":"local","query":"restaurants","location":"Miami","results":[{"business_id":"UXHxLN3DcDGI57uDIfCuJA","name":"Old\'s Havana Cuban Bar & Cocina","categories":["cuban","bars","venues"],"rating":4.4,"review_count":3217,"price":"$$","address":"[redacted-pii]","phone":"[redacted-pii]","distance_m":7039,"is_closed":false,"image_url":"https://s3-media0.fl.yelpcdn.com/bphoto/OyMD-xvBjobfDmDdBs2Jfw/o.jpg","yelp_url":"https://www.yelp.com/biz/olds-havana-cuban-bar-and-cocina-miami?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ","booking_url":"https://www.yelp.com/biz/olds-havana-cuban-bar-and-cocina-miami?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ","supports_reservations":false,"supports_delivery":false,"supports_pickup":true,"bookable":false},{"business_id":"oxtMfB

In [30]:
async def test_local_near_venue():
    async with streamable_http_client("https://mcp.agentorist.com/mcp") as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()

            result = await session.call_tool(
                "search",
                {
                    "vertical": "local",
                    "query": "restaurants",
                    "location": "New York",
                    "agent_client": "TripPlanner"
                }
            )

            print(result)

await test_local_near_venue()

meta=None content=[TextContent(type='text', text='{"vertical":"local","query":"restaurants","location":"New York","results":[{"business_id":"_EBQhV-UQ7NoV66uXYH0dw","name":"Golden Diner","categories":["diners","breakfast_brunch","sandwiches"],"rating":4.3,"review_count":1141,"price":"$$","address":"[redacted-pii]","phone":"[redacted-pii]","distance_m":765,"is_closed":false,"image_url":"https://s3-media0.fl.yelpcdn.com/bphoto/5c1hox9kAg_XzXFMAOqwZQ/o.jpg","yelp_url":"https://www.yelp.com/biz/golden-diner-new-york-2?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ","booking_url":"https://www.yelp.com/biz/golden-diner-new-york-2?adjust_creative=DzZmti2q5-VcwEdSD2XcyQ&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=DzZmti2q5-VcwEdSD2XcyQ","supports_reservations":false,"supports_delivery":true,"supports_pickup":true,"bookable":false},{"business_id":"hdiuRS9sVZSMReZm4oV5SA","name":"Da And

In [40]:
import requests
from bs4 import BeautifulSoup

html = requests.get("https://gribstream.com/mcp").text

for keyword in [
    "Authorization",
    "Bearer",
    "API",
    "apikey",
    "api-key",
    "token",
    "streamable",
    "mcp"
]:
    if keyword.lower() in html.lower():
        print("FOUND:", keyword)


FOUND: API
FOUND: token
FOUND: streamable
FOUND: mcp


In [41]:
from bs4 import BeautifulSoup

html = requests.get("https://gribstream.com/mcp").text
soup = BeautifulSoup(html, "html.parser")

text = soup.get_text("\n")

for line in text.splitlines():
    line = line.strip()
    if any(
        word in line.lower()
        for word in [
            "api",
            "key",
            "token",
            "authorization",
            "bearer",
            "mcp"
        ]
    ):
        print(line)

GribStream MCP Connector
OpenAPI
GribStream MCP connector
This is the hosted GribStream MCP endpoint for AI tools that support remote MCP over Streamable HTTP.
https://gribstream.com/mcp
The endpoint uses GribStream OAuth. After you sign in, you approve the connector and choose the active API token it should use for live
queries through the regular GribStream API. The raw API token is not shown to the AI client.
ChatGPT: configure a custom MCP connector using Streamable HTTP.
Gemini CLI: add this URL as an MCP server in
MCP announcement blog post
OpenAPI spec


In [5]:
from config.settings import settings

print("Weather Provider:", getattr(settings, "weather_provider", "<not set>"))
print("Weather MCP:", getattr(settings, "weather_mcp_server_url", "<not set>"))
print("Weather MCP:", settings.weather_mcp_server_url)

Weather Provider: livedatalink
Weather MCP: https://livedatalink.ai/mcp
Weather MCP: https://livedatalink.ai/mcp


In [6]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client
from config.settings import settings

async def test_kiwi_direct():

    payload = {
        "flyFrom": "Delhi",
        "flyTo": "Mumbai",
        "departureDate": "15/08/2026"
    }

    async with streamable_http_client(
        settings.kiwi_mcp_server_url
    ) as (
        read_stream,
        write_stream,
        _
    ):

        async with ClientSession(
            read_stream,
            write_stream
        ) as session:

            await session.initialize()

            result = await session.call_tool(
                "search-flight",
                payload
            )

            print(result)

await test_kiwi_direct()

meta=None content=[TextContent(type='text', text='[\n  {\n    "flyFrom": "DEL",\n    "flyTo": "BOM",\n    "cityFrom": "New Delhi",\n    "cityTo": "Mumbai",\n    "departure": {\n      "utc": "2026-08-14T22:30:00.000Z",\n      "local": "2026-08-15T04:00:00.000"\n    },\n    "arrival": {\n      "utc": "2026-08-15T00:45:00.000Z",\n      "local": "2026-08-15T06:15:00.000"\n    },\n    "totalDurationInSeconds": 8100,\n    "durationInSeconds": 8100,\n    "price": 64,\n    "deepLink": "https://on.kiwi.com/6Jx7Qu",\n    "currency": "EUR"\n  },\n  {\n    "flyFrom": "DEL",\n    "flyTo": "BOM",\n    "cityFrom": "New Delhi",\n    "cityTo": "Mumbai",\n    "departure": {\n      "utc": "2026-08-15T03:00:00.000Z",\n      "local": "2026-08-15T08:30:00.000"\n    },\n    "arrival": {\n      "utc": "2026-08-15T05:15:00.000Z",\n      "local": "2026-08-15T10:45:00.000"\n    },\n    "totalDurationInSeconds": 8100,\n    "durationInSeconds": 8100,\n    "price": 70,\n    "deepLink": "https://on.kiwi.com/sMa0fJ",

In [13]:
from pathlib import Path
import os

print("Current Working Directory:")
print(os.getcwd())

print("\nNotebook Path:")
print(Path.cwd())

Current Working Directory:
c:\Users\praya\Desktop\Tarvel_Agent\trip_planner\notebooks

Notebook Path:
c:\Users\praya\Desktop\Tarvel_Agent\trip_planner\notebooks


In [7]:
from pprint import pprint

print("=" * 80)
print("TRIP PLANNER MCP INTEGRATION TEST")
print("=" * 80)

# --------------------------------------------------
# FLIGHT MCP (KIWI)
# --------------------------------------------------

try:
    from tools.flight_tools import search_flights

    flights = search_flights(
        origin="Miami",
        destination="New York",
        event_date="2026-08-15",
        travelers=1,
    )

    print("\n✈️ FLIGHT MCP")
    print("Status:", flights.get("status"))

except Exception as e:
    print("\n❌ FLIGHT MCP ERROR")
    print(type(e).__name__, e)


# --------------------------------------------------
# HOTEL / LOCAL MCP (AGENTORIST)
# --------------------------------------------------

try:
    from tools.hotel_tools import search_hotels

    hotels = search_hotels(
        destination="New York"
    )

    print("\n🏨 HOTEL MCP")
    print("Status:", hotels.get("status"))

except Exception as e:
    print("\n❌ HOTEL MCP ERROR")
    print(type(e).__name__, e)


# --------------------------------------------------
# WEATHER MCP (LIVEDATALINK)
# --------------------------------------------------

try:
    from tools.weather_tools import get_weather

    weather = get_weather(
        destination="New York",
        event_date="2026-08-15",
    )

    print("\n🌦️ WEATHER MCP")
    print("Status:", weather.get("status"))

except Exception as e:
    print("\n❌ WEATHER MCP ERROR")
    print(type(e).__name__, e)


print("\n")
print("=" * 80)
print("SUMMARY")
print("=" * 80)

print("""
Expected:

Flight MCP  -> success
Hotel MCP   -> success
Weather MCP -> success
""")

TRIP PLANNER MCP INTEGRATION TEST
TOOL SCHEMA:
{'type': 'object', 'properties': {'flyFrom': {'type': 'string', 'minLength': 1, 'description': 'Location to fly from: It could be a city or an airport name or code'}, 'flyTo': {'type': 'string', 'description': 'Location to fly to: It could be a city or an airport name or code'}, 'departureDate': {'type': 'string', 'pattern': '^\\d{2}\\/\\d{2}\\/\\d{4}$', 'description': 'Departure date in dd/mm/yyyy format'}, 'departureDateFlexRange': {'type': 'integer', 'minimum': 0, 'maximum': 3, 'default': 0, 'description': 'Departure date flexibility range in days (0 to 3 days before/after the selected departure date)'}, 'returnDate': {'type': 'string', 'pattern': '^\\d{2}\\/\\d{2}\\/\\d{4}$', 'description': 'Return date in dd/mm/yyyy format'}, 'returnDateFlexRange': {'type': 'integer', 'minimum': 0, 'maximum': 3, 'default': 0, 'description': 'Return date flexibility range in days (0 to 3 days before/after the selected return date)'}, 'passengers': {'ty

In [5]:
from tools.flight_tools import search_flights
from pprint import pprint

result = search_flights(
    origin="Delhi",
    destination="Mumbai",
    event_date="2026-08-15",
)

print("\nRESULT TYPE:")
print(type(result))

print("\nRESULT:")
pprint(result)

TOOL SCHEMA:
{'type': 'object', 'properties': {'flyFrom': {'type': 'string', 'minLength': 1, 'description': 'Location to fly from: It could be a city or an airport name or code'}, 'flyTo': {'type': 'string', 'description': 'Location to fly to: It could be a city or an airport name or code'}, 'departureDate': {'type': 'string', 'pattern': '^\\d{2}\\/\\d{2}\\/\\d{4}$', 'description': 'Departure date in dd/mm/yyyy format'}, 'departureDateFlexRange': {'type': 'integer', 'minimum': 0, 'maximum': 3, 'default': 0, 'description': 'Departure date flexibility range in days (0 to 3 days before/after the selected departure date)'}, 'returnDate': {'type': 'string', 'pattern': '^\\d{2}\\/\\d{2}\\/\\d{4}$', 'description': 'Return date in dd/mm/yyyy format'}, 'returnDateFlexRange': {'type': 'integer', 'minimum': 0, 'maximum': 3, 'default': 0, 'description': 'Return date flexibility range in days (0 to 3 days before/after the selected return date)'}, 'passengers': {'type': 'object', 'properties': {'adu

In [9]:
from graph.trip_graph import build_trip_graph

print("Import Successful")

Import Successful


In [10]:
from graph.trip_graph import build_trip_graph

graph = build_trip_graph()

print(graph)

In [11]:
from graph.trip_graph import build_trip_graph

graph = build_trip_graph()

print(type(graph))

<class 'langgraph.graph.state.CompiledStateGraph'>


In [5]:
from tools.flight_tools import search_flights
from pprint import pprint

result = search_flights(
    origin="Miami",
    destination="New York",
    event_date="2026-08-15",
)

pprint(result)

TOOL SCHEMA:
{'type': 'object', 'properties': {'flyFrom': {'type': 'string', 'minLength': 1, 'description': 'Location to fly from: It could be a city or an airport name or code'}, 'flyTo': {'type': 'string', 'description': 'Location to fly to: It could be a city or an airport name or code'}, 'departureDate': {'type': 'string', 'pattern': '^\\d{2}\\/\\d{2}\\/\\d{4}$', 'description': 'Departure date in dd/mm/yyyy format'}, 'departureDateFlexRange': {'type': 'integer', 'minimum': 0, 'maximum': 3, 'default': 0, 'description': 'Departure date flexibility range in days (0 to 3 days before/after the selected departure date)'}, 'returnDate': {'type': 'string', 'pattern': '^\\d{2}\\/\\d{2}\\/\\d{4}$', 'description': 'Return date in dd/mm/yyyy format'}, 'returnDateFlexRange': {'type': 'integer', 'minimum': 0, 'maximum': 3, 'default': 0, 'description': 'Return date flexibility range in days (0 to 3 days before/after the selected return date)'}, 'passengers': {'type': 'object', 'properties': {'adu

In [4]:
from agents.flight_agent import flight_agent

state = {
    "origin": "Miami",
    "destination": "New York",
    "event_date": "2026-08-15",
    "travelers": 1,
    "errors": [],
}

result = flight_agent(state)

print("Flight Status:", result.get("flight_status"))
print("MCP Status:", result.get("flight_details", {}).get("status"))
print("Errors:", result.get("errors"))

TOOL SCHEMA:
{'type': 'object', 'properties': {'flyFrom': {'type': 'string', 'minLength': 1, 'description': 'Location to fly from: It could be a city or an airport name or code'}, 'flyTo': {'type': 'string', 'description': 'Location to fly to: It could be a city or an airport name or code'}, 'departureDate': {'type': 'string', 'pattern': '^\\d{2}\\/\\d{2}\\/\\d{4}$', 'description': 'Departure date in dd/mm/yyyy format'}, 'departureDateFlexRange': {'type': 'integer', 'minimum': 0, 'maximum': 3, 'default': 0, 'description': 'Departure date flexibility range in days (0 to 3 days before/after the selected departure date)'}, 'returnDate': {'type': 'string', 'pattern': '^\\d{2}\\/\\d{2}\\/\\d{4}$', 'description': 'Return date in dd/mm/yyyy format'}, 'returnDateFlexRange': {'type': 'integer', 'minimum': 0, 'maximum': 3, 'default': 0, 'description': 'Return date flexibility range in days (0 to 3 days before/after the selected return date)'}, 'passengers': {'type': 'object', 'properties': {'adu

In [3]:
from agents.itinerary_agent import itinerary_agent

state = {
    "destination": "New York",
    "venue": "Madison Square Garden",
    "event_date": "2026-08-15",
    "supervisor_notes": "Trip planning in progress.",
    "flight_notes": "Flight found from Miami to New York.",
    "hotel_notes": "Hotel found near Madison Square Garden.",
    "weather_notes": "Weather data retrieved successfully.",
    "search_notes": "Popular attractions identified.",
    "errors": [],
}

result = itinerary_agent(state)

print("Itinerary Status:", result.get("itinerary_status"))
print("Errors:", result.get("errors"))
print("Itinerary Present:", bool(result.get("itinerary")))

Itinerary Status: completed
Errors: []
Itinerary Present: True


In [4]:
from graph.trip_graph import build_trip_graph

graph = build_trip_graph()

result = graph.invoke(
    {
        "origin": "Miami",
        "destination": "New York",
        "travelers": 1,
        "venue": "Madison Square Garden",
        "event_date": "2026-08-15",
        "errors": [],
    }
)

print("status =", result.get("status"))
print("flight_status =", result.get("flight_status"))
print("hotel_status =", result.get("hotel_status"))
print("weather_status =", result.get("weather_status"))
print("itinerary_status =", result.get("itinerary_status"))
print("errors =", result.get("errors"))

RUNNING NODE: supervisor_agent

SUPERVISOR RECEIVED STATE
{'origin': 'Miami', 'destination': 'New York', 'travelers': 1, 'venue': 'Madison Square Garden', 'event_date': '2026-08-15', 'errors': []}

SUPERVISOR RETURNING STATE
{'origin': 'Miami', 'destination': 'New York', 'travelers': 1, 'venue': 'Madison Square Garden', 'event_date': '2026-08-15', 'errors': [], 'flight_details': {}, 'hotel_details': {}, 'weather_details': {}, 'search_results': {}, 'itinerary': '', 'flight_notes': '', 'hotel_notes': '', 'weather_notes': '', 'search_notes': '', 'itinerary_notes': '', 'execution_plan': {'run_flight_agent': True, 'run_hotel_agent': True, 'run_weather_agent': True, 'run_search_agent': True}, 'supervisor_notes': "**Supervisor Summary**\n\nThe current trip request is for a single traveler going from Miami to New York, with a specific venue in mind (Madison Square Garden) and a scheduled event date (2026-08-15). The execution plan involves running four agents: flight, hotel, weather, and searc

In [6]:
result = graph.invoke(state)

for k in sorted(result.keys()):
    if "status" in k:
        print(k, "=", result[k])

RUNNING NODE: supervisor_agent

SUPERVISOR RECEIVED STATE
{'destination': 'New York', 'venue': 'Madison Square Garden', 'event_date': '2026-08-15', 'flight_notes': 'Flight found from Miami to New York.', 'supervisor_notes': 'Trip planning in progress.', 'errors': []}

SUPERVISOR RETURNING STATE
{'destination': 'New York', 'venue': 'Madison Square Garden', 'event_date': '2026-08-15', 'flight_notes': 'Flight found from Miami to New York.', 'supervisor_notes': "**Supervisor Summary**\n\nThe current trip request is for a trip to New York, specifically to attend an event at Madison Square Garden on August 15, 2026. A flight from Miami to New York has already been found. The execution plan involves running four agents: flight, hotel, weather, and search.\n\n**Why the Execution Plan Makes Sense**\n\nThe execution plan makes sense because:\n\n1. **Flight Agent**: Since a flight from Miami to New York has already been found, running the flight agent again may help to confirm the details or expl

In [9]:
from state.trip_state import TripPlannerState

print("flight_status" in TripPlannerState.__annotations__)
print("hotel_status" in TripPlannerState.__annotations__)
print("weather_status" in TripPlannerState.__annotations__)
print("search_status" in TripPlannerState.__annotations__)
print("itinerary_status" in TripPlannerState.__annotations__)

True
True
True
True
True


In [3]:
from graph.trip_graph import build_trip_graph

graph = build_trip_graph()

result = graph.invoke(
    {
        "origin": "Miami",
        "destination": "New York",
        "travelers": 1,
        "venue": "Madison Square Garden",
        "event_date": "2026-08-15",
        "errors": [],
    }
)

print({
    "status": result.get("status"),
    "flight_status": result.get("flight_status"),
    "hotel_status": result.get("hotel_status"),
    "weather_status": result.get("weather_status"),
    "search_status": result.get("search_status"),
    "itinerary_status": result.get("itinerary_status"),
    "errors": len(result.get("errors", [])),
})

RUNNING NODE: supervisor_agent

SUPERVISOR RECEIVED STATE
{'origin': 'Miami', 'destination': 'New York', 'travelers': 1, 'venue': 'Madison Square Garden', 'event_date': '2026-08-15', 'errors': []}

SUPERVISOR RETURNING STATE
{'origin': 'Miami', 'destination': 'New York', 'travelers': 1, 'venue': 'Madison Square Garden', 'event_date': '2026-08-15', 'errors': [], 'flight_details': {}, 'hotel_details': {}, 'weather_details': {}, 'search_results': {}, 'itinerary': '', 'flight_notes': '', 'hotel_notes': '', 'weather_notes': '', 'search_notes': '', 'itinerary_notes': '', 'execution_plan': {'run_flight_agent': True, 'run_hotel_agent': True, 'run_weather_agent': True, 'run_search_agent': True}, 'supervisor_notes': "**Supervisor Summary**\n\nThe trip request is for a single traveler going from Miami to New York, with a specific venue in mind (Madison Square Garden) and a scheduled event date (2026-08-15). The current execution plan involves running all four agents: flight, hotel, weather, and s